# 03 - Benchmarking Models and Selecting Production Candidate

## Definition
Benchmarking compares multiple models under same split and metrics.

## Theory
Production selection should optimize tradeoff: quality + latency + maintainability.

## Motivation
Do not choose model by intuition. Use reproducible ranking with clear criteria.


In [ ]:
from pathlib import Path
import os

CWD = Path.cwd()
ROOT = CWD if (CWD / "pyproject.toml").exists() else CWD.parent
os.chdir(ROOT)
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/benchmarks").mkdir(parents=True, exist_ok=True)


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

bench_csv = Path("outputs/benchmarks/model_benchmark.csv")
automl_json = Path("outputs/benchmarks/automl_benchmark.json")

if not bench_csv.exists() or not automl_json.exists():
    subprocess.run([sys.executable, "scripts/train_model.py"], check=True)

benchmark_df = pd.read_csv(bench_csv)
benchmark_df.sort_values(["f1_macro", "accuracy"], ascending=False)


## Required Model Families
This table includes:
- Logistic Regression
- Decision Tree
- Random Forest
- Extra Trees
- XGBoost
- LightGBM
- CatBoost
- SVM
- KNN

Optional libraries load when installed.


In [ ]:
ranked = benchmark_df[benchmark_df["status"] == "ok"].sort_values(["f1_macro", "accuracy"], ascending=False)
ranked[["model_name", "accuracy", "precision_macro", "recall_macro", "f1_macro", "predict_time_ms"]]


In [ ]:
automl_df = pd.read_json("outputs/benchmarks/automl_benchmark.json")
automl_df


## LazyPredict vs FLAML vs PyCaret

### Why Each Exists
- **LazyPredict**: rapid baseline sweep
- **FLAML**: efficient AutoML search with time budget
- **PyCaret**: low-code experiment management and model comparison

### Strengths
- LazyPredict: speed and simplicity
- FLAML: strong cost-performance efficiency
- PyCaret: rich experiment abstraction

### Weaknesses
- LazyPredict: less control and deeper tuning
- FLAML: less pedagogical transparency than manual modeling
- PyCaret: heavier dependency stack

### Tradeoff
Use all three for teaching breadth, then pick production path based on control vs speed.


In [ ]:
bench_path = Path("outputs/benchmarks/model_benchmark_from_notebook03.csv")
ranked.to_csv(bench_path, index=False)
print(f"Saved: {bench_path}")

automl_path = Path("outputs/benchmarks/automl_benchmark_from_notebook03.json")
automl_path.write_text(automl_df.to_json(orient="records", indent=2), encoding="utf-8")
print(f"Saved: {automl_path}")
